In [17]:
import pandas as pd
from sentence_transformers import SentenceTransformer, util
import torch, json
from tqdm import tqdm

# ---- Paths ----
A_PATH = "/home/ubuntu/TW_MultiLabel_SMP/datasets/100_Round_3 - Final Annotations.csv"  # queries
B_PATH = "/home/ubuntu/TW_MultiLabel_SMP/datasets/400_Posts_Annotations - Combined_Dataset.csv"           # reference pool
A_EMB_PATH = "/home/ubuntu/embeddings/test_100_train_400_embeddings_A.pt"
B_EMB_PATH = "/home/ubuntu/embeddings/test_100_train_400_embeddings_B.pt"
SIM_JSON_OUT = "/home/ubuntu/TW_MultiLabel_SMP/similarity_scores/cross_similar_posts_100_train_400_test.json"

# ---- Read ----
A = pd.read_csv(A_PATH)
B = pd.read_csv(B_PATH)

# ---- Build full_text (same style as your notebook) ----
A['full_text'] = A['title'].fillna('') + '. ' + A['body'].fillna('')
B['full_text'] = B.get('title','').fillna('') + '. ' + B.get('body','').fillna('')  # robust to missing cols

A.head(), B.head()


(        id subreddit                                              title  \
 0  1lbqpe8  abortion                               Post abortion period   
 1  1lbqf4f  abortion  I feel so sad and lonely about having to abort...   
 2  1lbp51s  abortion                               Is buzz health safe?   
 3  1lbnzso  abortion                       SA after failed MA, Positive   
 4  1lbngxl  abortion                         overwhelmed & need to rant   
 
                                                 body         created_utc  \
 0  I had an abortion at roughly 5w5d. It was a me...  2025-06-15 3:29:57   
 1  TL;DR: This is not a viable pregnancy no matte...  2025-06-15 3:13:46   
 2  Well- i’m pregnant & i don’t want to be. I am ...  2025-06-15 2:04:00   
 3  Hi all,\n\nThrowaway for reasons but this page...  2025-06-15 1:01:37   
 4  Hello everyone, I found out I was pregnant abo...  2025-06-15 0:34:04   
 
                                                  url  \
 0  https://www.reddi

In [18]:
model = SentenceTransformer("all-mpnet-base-v2")

# Encode and save A
test4_emb_A = model.encode(
    A['full_text'].tolist(),
    convert_to_tensor=True,
    show_progress_bar=True,
    normalize_embeddings=True
)
torch.save(test4_emb_A, A_EMB_PATH)

# Encode and save B (do once; later you can just torch.load(B_EMB_PATH))
test4_emb_B = model.encode(
    B['full_text'].tolist(),
    convert_to_tensor=True,
    show_progress_bar=True,
    normalize_embeddings=True
)
torch.save(test4_emb_B, B_EMB_PATH)


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

In [19]:
emb_A = torch.load(A_EMB_PATH)
emb_B = torch.load(B_EMB_PATH)

In [20]:
k_max = 3

# Cosine similarity matrix: [len(A), len(B)]
# (normalize_embeddings=True above ⇒ cosine == dot)
sim_mat = util.cos_sim(test4_emb_A, test4_emb_B)  # torch tensor

# Top-k along B axis for each A row
top_vals, top_idx = torch.topk(sim_mat, k=k_max, dim=1)  # shapes: [len(A), k]

# Pack to dict: {a_row_index: [(b_index, score), ...]}
similar_posts = {}
for i in range(top_idx.size(0)):
    indices = top_idx[i].tolist()
    scores  = top_vals[i].tolist()
    similar_posts[i] = list(zip(indices, scores))

# Save JSON
with open(SIM_JSON_OUT, "w") as f:
    json.dump(similar_posts, f)

print(f"Saved cross-sim results to {SIM_JSON_OUT}")

Saved cross-sim results to /home/ubuntu/TW_MultiLabel_SMP/similarity_scores/cross_similar_posts_100_train_400_test.json
